# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 6: Erwerbstätigkeit ("Aufstocker")

Neues Themenfeld, direkt anschlussfähig an die Trichter-Grafik im Dashboard (zusätzliche Stufe „davon in Arbeit, aber bedürftig“). Bearbeitet Priorität 3 aus dem Priorisierungsvorschlag in [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md).

**Neue Datenquelle:** Anders als die bisherigen Notebooks liest dieses Notebook erstmals ein **neues Tabellenblatt** ein: „3.3 Erwerbstätigkeit“ (nicht in `eckwerte_long.csv` enthalten). Das Blatt enthält erwerbstätige erwerbsfähige Leistungsberechtigte (ELB) — in der öffentlichen Diskussion oft „Aufstocker“ genannt: Personen, die trotz eigenem Einkommen aus Beschäftigung ergänzend Bürgergeld beziehen.

**⚠️ Wichtige Besonderheit dieses Blatts:** Es liefert **zwei** Datenblöcke pro Datei — einen mit **3 Monaten Wartezeit** (gröbere Einkommens-Unterteilung: Geringfügigkeit / Übergangsbereich) und einen mit **6 Monaten Wartezeit** (feinere Unterteilung: Vollzeit/Teilzeit). Dieses Notebook nutzt ausschließlich den **3-Monate-Block**, da er aktueller ist. Da jede der 21 Dateien einen neuen 3-Monate-Block liefert, ergibt sich daraus eine durchgehende Zeitreihe — allerdings kalendarisch **3 Monate früher** als der Berichtsmonat der Datei: Aus den Dateien Januar 2025 – September 2026 ergibt sich eine Zeitreihe für **Oktober 2024 – Juni 2026**.

In [1]:
import re
from pathlib import Path

import openpyxl
import pandas as pd
import plotly.graph_objects as go

RAW_DIR = Path("..") / "Data" / "RegensburgJCData"
PROCESSED_DIR = Path("..") / "Data" / "processed"

MONATE = {
    "Januar": 1, "Februar": 2, "März": 3, "April": 4, "Mai": 5, "Juni": 6,
    "Juli": 7, "August": 8, "September": 9, "Oktober": 10, "November": 11, "Dezember": 12,
}

# Feste Zeilen im 3-Monate-Block des Blatts "3.3 Erwerbstätigkeit" (Spalte B = Wert)
ZEILEN = {
    36: "Insgesamt (ELB)",
    37: "erwerbstätige ELB",
    38: "abhängig erwerbstätig",
    39: "bis zur Geringfügigkeitsgrenze",
    40: "im Übergangsbereich",
    42: "über dem Übergangsbereich",
    43: "selbständig erwerbstätig",
}


def parse_monat(text):
    monat_name, jahr = text.split()
    return pd.Period(f"{jahr}-{MONATE[monat_name]:02d}", freq="M")


def parse_erwerbstaetigkeit(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["3.3 Erwerbstätigkeit"]
    monat = parse_monat(ws["B35"].value)
    zeile = {"period": monat}
    for r, label in ZEILEN.items():
        zeile[label] = ws.cell(row=r, column=2).value
    wb.close()
    return zeile


erwerbstaetigkeit = pd.DataFrame(
    [parse_erwerbstaetigkeit(p) for p in sorted(RAW_DIR.glob("*.xlsx"))]
).set_index("period").sort_index()
erwerbstaetigkeit.tail()

C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:33: UserWarning: DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.
  warn("DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.")
C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


,Insgesamt (ELB),erwerbstätige ELB,abhängig erwerbstätig,bis zur Geringfügigkeitsgrenze,im Übergangsbereich,über dem Übergangsbereich,selbständig erwerbstätig
period,,,,,,,
2026-02,3537,730,689,268,350,71,52
2026-03,3569,721,679,259,346,74,54
2026-04,3532,712,674,263,337,74,51
2026-05,3519,721,678,270,344,64,58
2026-06,3465,698,653,257,330,66,60


## 1. Datenüberblick und Plausibilitätsprüfung

Vor der eigentlichen Analyse: Ergibt sich wirklich eine lückenlose Zeitreihe, und stimmen die Teilsummen mit den Gesamtwerten überein?

In [2]:
print(f"Anzahl Monate: {len(erwerbstaetigkeit)}")
print(f"Zeitraum: {erwerbstaetigkeit.index.min()} bis {erwerbstaetigkeit.index.max()}")
print(f"Duplikate im Index: {erwerbstaetigkeit.index.duplicated().sum()}")

diff_erwerbstaetig = (
    erwerbstaetigkeit["abhängig erwerbstätig"] + erwerbstaetigkeit["selbständig erwerbstätig"]
    - erwerbstaetigkeit["erwerbstätige ELB"]
)
diff_einkommen = (
    erwerbstaetigkeit["bis zur Geringfügigkeitsgrenze"] + erwerbstaetigkeit["im Übergangsbereich"]
    + erwerbstaetigkeit["über dem Übergangsbereich"] - erwerbstaetigkeit["abhängig erwerbstätig"]
)
print(f"Differenz abhängig+selbständig vs. erwerbstätige ELB: min={diff_erwerbstaetig.min()}, max={diff_erwerbstaetig.max()}")
print(f"Differenz Einkommensstufen vs. abhängig erwerbstätig: min={diff_einkommen.min()}, max={diff_einkommen.max()}")

Anzahl Monate: 21
Zeitraum: 2024-10 bis 2026-06
Duplikate im Index: 0
Differenz abhängig+selbständig vs. erwerbstätige ELB: min=4, max=15
Differenz Einkommensstufen vs. abhängig erwerbstätig: min=0, max=0


Alle 21 Dateien liefern erwartungsgemäß 21 unterschiedliche, lückenlos aufeinanderfolgende Monate (Oktober 2024 – Juni 2026, keine Duplikate). Die Einkommensstufen summieren sich exakt zur Zahl der abhängig Erwerbstätigen (Differenz = 0 in jedem Monat) — die Aufteilung nach Geringfügigkeitsgrenze und Übergangsbereich ist also vollständig. Bei „abhängig + selbständig erwerbstätig“ gegenüber „erwerbstätige ELB“ bleibt dagegen in jedem Monat eine kleine Differenz von 4 bis 15 Personen (nicht eindeutig zugeordnete Fälle) — ein kleiner, aber über alle 21 Monate konsistenter Rundungs- bzw. Erhebungseffekt der Bundesagentur, kein Fehler in der eigenen Auswertung.

## 2. Erwerbstätige ELB im Zeitverlauf: die Aufstocker-Quote

Die **Aufstocker-Quote** (nicht direkt in den Rohdaten enthalten, hier selbst berechnet) setzt die erwerbstätigen ELB ins Verhältnis zu allen ELB: Wie viele der erwerbsfähigen Leistungsberechtigten haben zusätzlich eigenes Erwerbseinkommen?

In [3]:
erwerbstaetigkeit["Aufstockerquote (%)"] = (
    erwerbstaetigkeit["erwerbstätige ELB"] / erwerbstaetigkeit["Insgesamt (ELB)"] * 100
).round(1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=erwerbstaetigkeit.index.to_timestamp(), y=erwerbstaetigkeit["erwerbstätige ELB"], name="erwerbstätige ELB (absolut)",
    mode="lines+markers", line=dict(color="#2563eb"), marker=dict(size=6), yaxis="y1",
))
fig.add_trace(go.Scatter(
    x=erwerbstaetigkeit.index.to_timestamp(), y=erwerbstaetigkeit["Aufstockerquote (%)"], name="Aufstockerquote (%)",
    mode="lines+markers", line=dict(color="#f59e0b", dash="dot"), marker=dict(size=6), yaxis="y2",
))
fig.update_layout(
    title="Erwerbstätige ELB (\"Aufstocker\") im Zeitverlauf, Oktober 2024 – Juni 2026 (SGB II)",
    yaxis=dict(title="Personen"),
    yaxis2=dict(title="Anteil an allen ELB (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis_title="Monat (3 Monate vor dem jeweiligen Berichtsmonat)",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=460,
)
fig

Die Zahl der erwerbstätigen ELB ist von 795 (Oktober 2024) auf 698 (Juni 2026) gesunken — ein Rückgang, der zeitlich mit dem insgesamt sinkenden ELB-Bestand zusammenfällt (vgl. Rückgang der Bedarfsgemeinschaften in [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb)). Die Aufstockerquote selbst ist mit 20,1 % bis 22,3 % (Ø 21,2 %) bemerkenswert stabil und sinkt nur leicht von 21,8 % auf 20,1 % (Juni 2026, bisheriger Tiefstwert) — **rund jede fünfte erwerbsfähige leistungsberechtigte Person im SGB-II-Bezug arbeitet also bereits, verdient aber nicht genug, um ohne ergänzendes Bürgergeld auszukommen.**

## 3. Struktur der erwerbstätigen ELB: abhängig vs. selbständig

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=erwerbstaetigkeit.index.to_timestamp(), y=erwerbstaetigkeit["abhängig erwerbstätig"], name="abhängig erwerbstätig",
    mode="lines", stackgroup="erwerb", line=dict(color="#2563eb", width=0.5),
))
fig.add_trace(go.Scatter(
    x=erwerbstaetigkeit.index.to_timestamp(), y=erwerbstaetigkeit["selbständig erwerbstätig"], name="selbständig erwerbstätig",
    mode="lines", stackgroup="erwerb", line=dict(color="#7c3aed", width=0.5),
))
fig.update_layout(
    title="Erwerbstätige ELB: abhängig vs. selbständig erwerbstätig",
    yaxis_title="Personen",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=420,
)
fig

Die Struktur ist über den gesamten Zeitraum stabil: **abhängig erwerbstätig** dominiert klar mit 92,9 % bis 94,7 % der erwerbstätigen ELB (Oktober 2024: 93,3 %, Juni 2026: 93,6 %), **selbständig erwerbstätig** bleibt eine kleine Minderheit von rund 6–9 % — mit zuletzt leicht steigender absoluter Zahl (51 im April 2026, 60 im Juni 2026). Anders als bei der Langzeitarbeitslosigkeit (siehe [`03_Statistik.ipynb`](03_Statistik.ipynb)) zeigt sich hier keine auffällige Verschiebung — die Erwerbsform der Aufstocker bleibt konstant.

## 4. Einkommensstruktur der abhängig Erwerbstätigen

Drei sich nicht überschneidende Einkommensstufen (Summe = „abhängig erwerbstätig“): **bis zur Geringfügigkeitsgrenze** (derzeit 603 €), **im Übergangsbereich** (bis 2.000 €) und **über dem Übergangsbereich**.

In [5]:
einkommens_stufen = ["bis zur Geringfügigkeitsgrenze", "im Übergangsbereich", "über dem Übergangsbereich"]
farben_einkommen = {"bis zur Geringfügigkeitsgrenze": "#f59e0b", "im Übergangsbereich": "#2563eb", "über dem Übergangsbereich": "#16a34a"}

fig = go.Figure()
for stufe in einkommens_stufen:
    fig.add_trace(go.Scatter(
        x=erwerbstaetigkeit.index.to_timestamp(), y=erwerbstaetigkeit[stufe], name=stufe,
        mode="lines", stackgroup="einkommen", line=dict(color=farben_einkommen[stufe], width=0.5),
    ))
fig.update_layout(
    title="Einkommensstruktur der abhängig erwerbstätigen ELB",
    yaxis_title="Personen",
    xaxis_title="Monat",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.15),
    height=420,
)
fig

Innerhalb der abhängig Erwerbstätigen verschiebt sich das Bild leicht in Richtung höherer Einkommen: Der Anteil **bis zur Geringfügigkeitsgrenze** sinkt von 43,4 % (Oktober 2024) auf 39,4 % (Juni 2026; Tiefstwert 38,1 % im März 2026), während **im Übergangsbereich** von 48,0 % auf 50,5 % und **über dem Übergangsbereich** von 8,6 % auf 10,1 % steigen. In den letzten drei Monaten hat sich die Verschiebung leicht abgeschwächt. Auch wenn die absoluten Zahlen insgesamt rückläufig sind (siehe Kapitel 3), verdienen die verbleibenden Aufstocker im Schnitt tendenziell mehr.

## 5. Export der aufbereiteten Daten

Grundlage für den Dashboard-Tab „💼 Erwerbstätigkeit“.

In [6]:
erwerbstaetigkeit.to_csv(PROCESSED_DIR / "erwerbstaetigkeit_sgb2.csv")
erwerbstaetigkeit.tail()

,Insgesamt (ELB),erwerbstätige ELB,abhängig erwerbstätig,bis zur Geringfügigkeitsgrenze,im Übergangsbereich,über dem Übergangsbereich,selbständig erwerbstätig,Aufstockerquote (%)
period,,,,,,,,
2026-02,3537,730,689,268,350,71,52,20.6
2026-03,3569,721,679,259,346,74,54,20.2
2026-04,3532,712,674,263,337,74,51,20.2
2026-05,3519,721,678,270,344,64,58,20.5
2026-06,3465,698,653,257,330,66,60,20.1


## 6. Zusammenfassung

- **Aufstockerquote:** stabil bei Ø 21,2 % (20,1–22,3 %) — rund jede fünfte erwerbsfähige leistungsberechtigte Person im SGB-II-Bezug ist erwerbstätig, verdient aber nicht genug für den vollständigen Ausstieg aus dem Bürgergeld-Bezug. Leicht rückläufiger Trend von 21,8 % auf 20,1 %.
- **Erwerbsform:** klar dominiert von abhängiger Beschäftigung (92,9–94,7 %), selbständig Erwerbstätige bleiben eine stabile Minderheit von ~6–9 %.
- **Einkommensstruktur:** leichte Verschiebung zu höheren Einkommen — der Anteil „bis zur Geringfügigkeitsgrenze“ sinkt (43,4 % → 39,4 %), während „im Übergangsbereich“ (48,0 % → 50,5 %) und „über dem Übergangsbereich“ (8,6 % → 10,1 %) steigen.
- **Datenbesonderheit:** Diese Zeitreihe ist gegenüber dem Berichtsmonat der Rohdaten um 3 Monate versetzt (Oktober 2024 – Juni 2026 statt Januar 2025 – September 2026) — ein direkter Vergleich mit den übrigen Kennzahlen ist daher nur mit diesem Versatz im Hinterkopf sinnvoll.

Insgesamt zeigt sich: Erwerbstätigkeit trotz Bürgergeld-Bezug ist kein Randphänomen, sondern betrifft konstant gut ein Fünftel aller erwerbsfähigen Leistungsberechtigten — mit einer leicht positiven Tendenz hin zu höheren Einkommen bei insgesamt sinkender absoluter Zahl.

Weiter geht es mit der Umsetzung im [Streamlit-Dashboard](../Dashboard/app.py) (Tab „💼 Erwerbstätigkeit“) sowie optional mit `07_Finanzen.ipynb` (Zahlungsansprüche, ⚠️ 3 Monate Wartezeit).